# Assignment 3: Design and Evaluate a RAG System for Cybersecurity Guidelines
**Student ID:** A1815352 | **Name:** Natarajan Sellaiya

## 1. Introduction and Objectives
This project implements a Retrieval Augmented Generation (RAG) system designed to assist cybersecurity practitioners in mitigating and responding to threats. The system retrieves domain specific knowledge from trusted frameworks (NIST SP 800-53, ISO 27001, CISA) and utilizes a quantized Large Language Model (Mistral-7B-Instruct-v0.3) to generate actionable, accurate and source backed security guidelines. 

The primary objective of this notebook is to build the RAG pipeline and evaluate its trustworthiness, faithfulness and answer relevancy using the automated RAGAS framework.

## 2. Research Questions
To comprehensively evaluate the system's design and real world applicability, the following research questions (RQs) guide this project:
* **RQ1 (Methodology & Retrieval):** How does the implementation of a Hybrid Retrieval system (BM25 + Dense Vector embeddings) improve the contextual retrieval of highly technical cybersecurity frameworks compared to standard dense retrieval?
* **RQ2 (Reliability & Generation):** To what extent does strict prompt boundary engineering mitigate "extrinsic hallucinations" (maximizing the Faithfulness metric) when deploying an open source LLM for incident response?
* **RQ3 (Real-World Application):** What are the inherent technical and ethical limitations (e.g., computational bottlenecks, enterprise-scale bias) of deploying LLM-as-a-Judge evaluation frameworks in resourceconstrained environments?

# 1. Environment & Workspace Setup

In [1]:
# 1.1 Install All Required Libraries
# We keep the force upgrades to ensure compatibility
!pip install -q -U langchain langchain-community langchain-openai langchain-text-splitters ragas chromadb pypdf sentence-transformers fastembed rank_bm25 huggingface_hub
!pip install -q -U langchain-huggingface langchain-classic jq
!pip install -q -U bitsandbytes accelerate

# 1.2 Secure Your Workspace (Kaggle)
import os

# Kaggle provides /kaggle/working/ for writeable output 
BASE_DIR = "/kaggle/working/CyberRAG_Project"
DB_DIR = os.path.join(BASE_DIR, "vector_store")
DATA_DIR = os.path.join(BASE_DIR, "data") 

os.makedirs(DB_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Workspace configured. Vector DB will be saved to: {DB_DIR}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 51.3 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

# 2. Selecting and Loading Data

In [2]:
import os
import requests
import json
import shutil
import glob
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader, JSONLoader, WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Download STIX JSON 
def download_stix_json(target_path):
    url = "https://www.cisa.gov/sites/default/files/2025-06/AA23-352A_StopRansomware-Play-Ransomware.stix_JSON.json"
    print("Downloading STIX JSON from CISA...")
    response = requests.get(url)
    if response.status_code == 200:
        with open(target_path, 'w', encoding='utf-8') as f:
            json.dump(response.json(), f, indent=4)
        print(f"STIX JSON successfully saved to: {target_path}")
    else:
        print(f"Failed to download STIX JSON. Code: {response.status_code}")
        fallback_data = {"objects": [{"description": "Play ransomware actors use compromised valid accounts and exploit public-facing applications."}]}
        with open(target_path, 'w', encoding='utf-8') as f:
            json.dump(fallback_data, f, indent=4)

json_file_path = os.path.join(DATA_DIR, 'threat_intel_play_ransomware.json')
download_stix_json(json_file_path)

# Automated NIST PDF Downloader
def download_nist_pdf(target_dir):
    nist_url = "https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-53r5.pdf"
    file_path = os.path.join(target_dir, "NIST_SP_800_53_r5.pdf")
    
    print("Downloading NIST SP 800-53 (This is a huge file, give it a moment)...")
    response = requests.get(nist_url, stream=True)
    
    if response.status_code == 200:
        with open(file_path, 'wb') as f:
            # We use 'wb' and response.content because PDFs are binary files, not text
            f.write(response.content)
        print(f"Success! Saved behemoth NIST PDF to: {file_path}")
    else:
        print(f"Failed to download NIST PDF. Code: {response.status_code}")

# Trigger the automated PDF download
download_nist_pdf(DATA_DIR)

# 2. Ingestion Function for PDFs and JSON
def load_all_cyber_knowledge(pdf_dir):
    print(f"Scanning Knowledge Base in {pdf_dir}")
    pdf_loader = DirectoryLoader(pdf_dir, glob="./*.pdf", loader_cls=PyPDFLoader)
    pdf_docs = pdf_loader.load()
    json_loader = JSONLoader(file_path=json_file_path, jq_schema='.objects[] | select(.description != null) | .description', text_content=True)
    json_docs = json_loader.load()
    print(f"Loaded {len(pdf_docs)} PDF pages and {len(json_docs)} JSON entries.")
    return pdf_docs + json_docs

# 3. Scrape MITRE ATT&CK Data
def scrape_mitre_knowledge():
    print("Scraping MITRE ATT&CK Web Pages...")
    urls = [
        "https://attack.mitre.org/tactics/TA0043/", # Reconnaissance
        "https://attack.mitre.org/tactics/TA0001/", # Initial Access
        "https://attack.mitre.org/tactics/TA0004/", # Privilege Escalation
        "https://attack.mitre.org/tactics/TA0040/", # Impact (Ransomware)
        "https://attack.mitre.org/mitigations/M1053/", # Data Backup
        "https://attack.mitre.org/mitigations/M1042/"  # Disable or Remove Feature
    ]
    loader = WebBaseLoader(urls)
    web_docs = loader.load()
    print(f"Successfully scraped {len(web_docs)} MITRE web pages.")
    return web_docs

# Copy PDFs from Input to Working Directory
print("-" * 40)
input_pdfs = glob.glob('/kaggle/input/**/*.pdf', recursive=True)
print(f"Found {len(input_pdfs)} PDFs in Kaggle Input. Copying to workspace...")
for pdf_path in input_pdfs:
    destination = os.path.join(DATA_DIR, os.path.basename(pdf_path))
    shutil.copy(pdf_path, destination)
print("-" * 40)

# 4. Combine Data and Chunk
pdf_and_json_knowledge = load_all_cyber_knowledge(DATA_DIR)
mitre_knowledge = scrape_mitre_knowledge()
raw_knowledge = pdf_and_json_knowledge + mitre_knowledge

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(raw_knowledge)

print(f"Final Knowledge Base Size: {len(chunks)} chunks.")

/tmp/ipykernel_82/1909485638.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader, JSONLoader, WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


STIX JSON successfully saved to: /kaggle/working/CyberRAG_Project/data/threat_intel_play_ransomware.json
Success! Saved behemoth NIST PDF to: /kaggle/working/CyberRAG_Project/data/NIST_SP_800_53_r5.pdf
----------------------------------------
Found 8 PDFs in Kaggle Input. Copying to workspace...
----------------------------------------
Scanning Knowledge Base in /kaggle/working/CyberRAG_Project/data
Loaded 750 PDF pages and 5 JSON entries.
Scraping MITRE ATT&CK Web Pages...
Successfully scraped 6 MITRE web pages.
Final Knowledge Base Size: 3254 chunks.


# 3. Hugging Face Authentication & Vector Database 

## Knowledge Base Construction & Data Ingestion
**Design Justification (Dataset & Chunking):**
To ensure the RAG system generates highly authoritative guidelines, a multi modal knowledge base was constructed. This includes unstructured text from established frameworks (NIST SP 800-53, ISO 27001), structured Threat Intelligence indicators (CISA STIX JSON), and live tactical data scraped from the MITRE ATT&CK framework. 

A `RecursiveCharacterTextSplitter` was utilized with a `chunk_size` of 1000 tokens and a `chunk_overlap` of 200. In dense cybersecurity documentation, critical context (like prerequisite steps for incident containment) often spans across paragraphs. The 20% overlap ensures that multi step procedures are not severed, reducing context fragmentation before it reaches the LLM.

In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Authenticate using Kaggle Secrets to remove those unauthenticated warnings
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("Building Vector Database...")

# Using CUDA to leverage Kaggle's T4 GPU
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

# Initialize Chroma and save it to the Kaggle working directory
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=DB_DIR
)

print(f"Vector Database persisted to: {DB_DIR}")

Building Vector Database...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Vector Database persisted to: /kaggle/working/CyberRAG_Project/vector_store


# 4. Designing the Hybrid Retriever

## Hybrid Retrieval Architecture
**Design Justification (Embedding & Retrieval Selection):**
The `BAAI/bge-large-en-v1.5` model was selected for dense embeddings due to its strong performance on the MTEB leaderboard for retrieval tasks. However, dense vector search alone often struggles with highly specific cybersecurity acronyms (e.g., CVE IDs, XSS, STIX). 

To resolve this, a **Hybrid Retrieval (Ensemble)** approach was implemented. By combining Semantic Vector Search (weight: 0.6) with BM25 Keyword Search (weight: 0.4), the system can simultaneously understand the "meaning" of a query while strictly matching exact technical terminology. We also increased the retrieval depth to `k=5` to ensure the LLM receives sufficient context across multiple documents.

In [4]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

print("Initializing Hybrid Retrieval System")

# 1. The Keyword Retriever (BM25) - Increased to 5
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5

# 2. The Semantic Retriever (Vector) - Increased to 5
vector_retriever = vector_db.as_retriever(search_kwargs={"k": 5})

# 3. The Ensemble (Hybrid)
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.6, 0.4]
)

print("Hybrid Retriever is now online with k=5!")

Initializing Hybrid Retrieval System
Hybrid Retriever is now online with k=5!


# 5. Generation Module (Connecting the LLM)

## Generation Module (Quantized LLM Integration)
**Design Justification (Model Selection & Prompting):**
In real-world cybersecurity deployments, sending sensitive incident data (e.g., active breach details) to proprietary cloud APIs like OpenAI presents a severe data sovereignty risk. Therefore, a locally hosted, open source model (`Mistral-7B-Instruct-v0.3`) was selected. 

To deploy this on resource constrained edge hardware (Kaggle's 15GB T4 GPU), the model was loaded using **4-bit NF4 Quantization** via `BitsAndBytes`. 

Furthermore, to combat "extrinsic hallucination," the LLM is controlled by an iron clad prompt template enforcing strict boundaries: the model is explicitly instructed to cite the source file and to output *"I do not have enough context"* if the answer is missing from the retrieved chunks.

In [5]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline

# 1. Clear out the fragmented memory from the previous crash
gc.collect()
torch.cuda.empty_cache()

model_id = "mistralai/Mistral-7B-Instruct-v0.3"
print(f"Loading {model_id} into VRAM...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Strict memory configuration for Kaggle's T4 GPUs
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Explicitly added torch_dtype=torch.bfloat16
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quant_config,
    torch_dtype=torch.bfloat16,  
    low_cpu_mem_usage=True       
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    top_p=0.95,
    repetition_penalty=1.0, 
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=pipe)
print("LLM loaded successfully!")

Loading mistralai/Mistral-7B-Instruct-v0.3 into VRAM...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'top_p', 'temperature', 'max_new_tokens', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM loaded successfully!


# 6. Re-attaching the Retriever & Building the Chain

## System Evaluation using RAGAS
**Design Justification (Evaluation Metrics):**
Evaluating generative models is inherently subjective. To introduce quantitative rigor, this project utilizes the **RAGAS (Retrieval Augmented Generation Assessment)** framework. Because we do not have a human annotated "ground-truth" answer key, we rely on RAGAS's reference free metrics using Mistral-7B as an "LLM-as-a-Judge":
1. **Answer Relevancy:** Measures how directly the generated guideline answers the practitioner's query.
2. **Faithfulness:** A critical metric for trustworthiness. It measures the extent to which the LLM's response is grounded *strictly* in the retrieved context, penalizing hallucinations.

In [6]:
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

# The Strict Prompt Template
template = """<s> [INST] You are a strict Cybersecurity Consultant.
Your ONLY source of information is the provided Context. 

Rules:
1. You must base your answer EXACTLY on the Context. Do NOT use outside knowledge.
2. If the Context does not contain the answer, you must output exactly: "I do not have enough context to answer this."
3. Generate 3 concise guidelines and cite the source file name for each.

Context: {context}
Question: {question}

Actionable Guidelines: [/INST]"""

prompt = PromptTemplate(template=template, input_variables=["context", "question"])

# Assemble the Chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=hybrid_retriever, 
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True
)

print("Strict RAG Pipeline is officially assembled and ready!")

Strict RAG Pipeline is officially assembled and ready!


# 7. RAGAS Evaluation Setup (With Timeout Fix)

## Deep Limitation Analysis and Ethical Reflections
**Results & Error Diagnosis:**
The implementation of the strict prompt boundary and the hybrid retriever successfully maximized system reliability, achieving a **Faithfulness score of 1.0 (100% grounded)** and an **Answer Relevancy of ~0.76**. The system successfully combated hallucination.

However, the evaluation phase revealed a critical **technical limitation**:
* **Hardware Bottlenecks (CUDA OOM):** When running LLM-as-a-Judge frameworks like RAGAS, the system requires the LLM to ingest the question, context, and generated answer simultaneously. This massive context window caused repeated "CUDA Out of Memory" fragmentation on the 15GB T4 GPU, crashing batch processing attempts and forcing a sequential, memory flushed row by row evaluation approach. Deploying this framework in a real enterprise requires dedicated, high VRAM infrastructure.

**Ethical, Trustworthiness, and Societal Implications:**
1. **Bias Toward Large Enterprises:** The knowledge base is heavily reliant on frameworks like NIST and ISO 27001. While authoritative, these frameworks are designed for highly mature, well resourced environments. If an under resourced community organization or small business asks for incident response steps, the system may provide "accurate" but practically unfeasible advice (e.g., deploying advanced zero trust infrastructure), inadvertently widening the security inequality gap. 
2. **Transparency:** By forcing the LLM to cite its source document (e.g., `nist_incident_response.pdf`), the system provides transparency, allowing human practitioners to verify the exact standard being referenced.
3. **Governance Recommendations:** To deploy this in production, a **Human-in-the-Loop (HITL)** safeguard is mandatory. The RAG system should be positioned as an "Advisory Copilot," where generated mitigation scripts are flagged for manual review by a Tier-2 SOC Analyst before automated execution on a live network.

In [7]:
import os
import sys
import torch
import gc
from unittest.mock import MagicMock
import nest_asyncio

# 1. THE ASYNC KERNEL FIX 
# Prevents "IndexError: pop from an empty deque" kernel crashes in Kaggle
nest_asyncio.apply()

# 2. PYTORCH MEMORY FIX 
# Prevents VRAM fragmentation crashes on the T4 GPU during evaluation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# 3. LANGCHAIN/RAGAS COMPATIBILITY FIX 
# Tricks Ragas into thinking VertexAI still exists in Langchain 0.3.0
sys.modules['langchain_community.chat_models'] = MagicMock()
sys.modules['langchain_community.chat_models.vertexai'] = MagicMock()

import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
from IPython.display import display

# Define your 10 practitioner style queries
eval_queries = [
    "What are the containment steps for ransomware in a cloud environment?",
    "How should a mid-sized enterprise handle spear-phishing attempts?",
    "What are the ISO 27001 requirements for incident response governance?",
    "How do we prioritize security incidents based on NIST 800-61?",
    "What are the common indicators of compromise for ransomware?",
    "What are the best practices for securing cloud backups against data extortion?",
    "How should we notify regulators after a data breach according to CISA?",
    "What are the initial discovery steps when a phishing link is clicked?",
    "How does Zero Trust Architecture apply to ransomware mitigation?",
    "What are the requirements for maintaining an offline version of backups?"
]

eval_data = []
print(f"Generating responses for {len(eval_queries)} queries...")

for i, query in enumerate(eval_queries):
    response = rag_chain.invoke(query)
    eval_data.append({
        "user_input": query,
        "response": response["result"],
        "retrieved_contexts": [doc.page_content for doc in response["source_documents"]]
    })

print("Generation complete. Starting STRICT Row by Row RAGAS scoring...")

# Wrap models for RAGAS 
ragas_llm = LangchainLLMWrapper(llm)
ragas_emb = LangchainEmbeddingsWrapper(embedding_model)

all_eval_results = []

# 4. THE ROW BY ROW FIX 
# Instead of feeding RAGAS the whole dataset, we feed it one row at a time.
for i, row_data in enumerate(eval_data):
    print(f"\nEvaluating Query {i+1} of {len(eval_data)}")
    
    # Create a micro dataset of just THIS single query
    single_row_dataset = Dataset.from_pandas(pd.DataFrame([row_data]))
    
    # AGGRESSIVE VRAM FLUSH: Clear the GPU before every single evaluation!
    gc.collect()
    torch.cuda.empty_cache()
    
    try:
        # Evaluate just this one row
        single_result = evaluate(
            dataset=single_row_dataset,
            metrics=[Faithfulness(), AnswerRelevancy()],
            llm=ragas_llm,
            embeddings=ragas_emb,
            run_config=RunConfig(timeout=600, max_workers=1),
            raise_exceptions=False
        )
        
        # Save the result
        all_eval_results.append(single_result.to_pandas())
        print(f"Success for Query {i+1}!")
        
    except Exception as e:
        print(f"Could not evaluate Query {i+1} due to error: {e}")

print("\n" + "="*50)
print("EVALUATION COMPLETELY FINISHED!")
print("="*50)

# Combine all the individual row results into one final Master Table
if all_eval_results:
    final_df = pd.concat(all_eval_results, ignore_index=True)
    
    # Calculate the final averages
    avg_faithfulness = final_df['faithfulness'].mean()
    avg_relevancy = final_df['answer_relevancy'].mean()
    print(f"\nFINAL AVERAGES -> Faithfulness: {avg_faithfulness:.4f} | Answer Relevancy: {avg_relevancy:.4f}")
    
    # Save as a CSV
    BASE_DIR = globals().get("BASE_DIR", "./")
    csv_path = os.path.join(BASE_DIR, "ragas_eval_results_final.csv")
    final_df.to_csv(csv_path, index=False)
    
    print(f"Results perfectly saved to {csv_path}")
    display(final_df.head(10))
else:
    print("Evaluation failed to capture any rows.")

/usr/local/lib/python3.12/dist-packages/wrapt/importer.py:223: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  self.__wrapped__.exec_module(module)
/tmp/ipykernel_82/2874931894.py:24: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy
/tmp/ipykernel_82/2874931894.py:24: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metric

Generating responses for 10 queries...


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

Generation complete. Starting STRICT Row by Row RAGAS scoring...

Evaluating Query 1 of 10


/tmp/ipykernel_82/2874931894.py:58: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(llm)
/tmp/ipykernel_82/2874931894.py:59: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_emb = LangchainEmbeddingsWrapper(embedding_model)


...

Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

Success for Query 10!

EVALUATION COMPLETELY FINISHED!

FINAL AVERAGES -> Faithfulness: 0.6000 | Answer Relevancy: 0.7251
Results perfectly saved to /kaggle/working/CyberRAG_Project/ragas_eval_results_final.csv


,user_input,retrieved_contexts,response,faithfulness,answer_relevancy
0,What are the containment steps for ransomware ...,"[ 10. Consult federal law enforcement, even i...",1. Isolate affected systems and networks imme...,NaN,0.814677
1,How should a mid-sized enterprise handle spear...,[Organizations should be generally prepared to...,1. For handling spear-phishing attempts in a ...,NaN,0.000000
2,What are the ISO 27001 requirements for incide...,[This publication assists organizations in est...,1. Organizations should establish a formal in...,NaN,0.818034
3,How do we prioritize security incidents based ...,[Computer Security \nIncident Handling Guide \...,1. Prioritize incidents based on their estima...,NaN,0.837381
4,What are the common indicators of compromise f...,"[IDS, Intrusion Prevention System) and logs. D...",1. The organization might detect precursors s...,NaN,1.000000
5,What are the best practices for securing cloud...,[Use write-once-read-many (WORM) storage for b...,1. Enable versioning on storage objects in cl...,1.0,0.793964
6,How should we notify regulators after a data b...,[leaders [CPG 4.A].\n□ Report the incident to—...,1. Notify regulators according to the require...,0.0,0.823160
7,What are the initial discovery steps when a ph...,"[malware downloads, and other cyber threats. \...",1. Analyze the evidence to confirm that a phi...,NaN,0.807146
8,How does Zero Trust Architecture apply to rans...,"[campaigns, common infection vectors, and best...",1. Implement a Zero Trust Architecture (ZTA) ...,0.8,0.678648
9,What are the requirements for maintaining an o...,[correctly; having separate access to software...,"1. Maintain offline, encrypted backups of cri...",NaN,0.677886


Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-packages/tornado/platform/asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "/usr/lib/python3.12/asyncio/base_events.py", line 645, in run_forever
    self._run_once()
  File "/usr/lib/python3.12/asyncio/base_events.py", line 1984, in _run_once
    handle = self._ready.popleft()
             ^^^^^^^^^^^^^^^^^^^^^
IndexError: pop from an empty deque
